In [2]:
import numpy as np
from single_head_attention import attention

In [6]:
def multihead_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    Q = np.dot(X, W_Q)
    V = np.dot(X, W_V)
    K = np.dot(X, W_K)

    dim_k = K.shape[-1] // num_heads

    list_Q = []
    list_V = []
    list_K = []

    for i in range(0, Q.shape[1], dim_k):
        list_Q.append(Q[:, i:i+dim_k])
        list_K.append(K[:, i:i+dim_k])
        list_V.append(V[:, i:i+dim_k])

    slices_with_attention = []

    for i in range(len(list_Q)):
        slices_with_attention.append(attention(list_Q[i], list_K[i], list_V[i]))

    concatenated = np.concatenate(slices_with_attention, axis=1)

    return np.dot(concatenated, W_O)

In [7]:
def ReLU(layer):
    return np.maximum(0, layer)

def feed_forward(X, W1, b1, W2, b2):
    inner_layer = ReLU(np.dot(X, W1) + b1)

    return np.dot(inner_layer, W2) + b2

In [5]:
def layer_norm(output_token, gamma, beta, eps=1e-6):
    mean = np.mean(output_token)
    standard_deviation = np.std(output_token)

    main_calculation = np.divide(np.subtract(output_token, mean), np.add(standard_deviation, eps))

    return np.add(np.multiply(gamma, main_calculation), beta)

In [10]:
def transformer_block(X, Q_0, K_0, V_0, W_0, W1, b1, W2, b2, gamma1, beta1, gamma2, beta2):
    multi_head = multihead_attention(X, Q_0, K_0, V_0, W_0, 2)

    first_residual = X + multi_head
    first_layer_norm = layer_norm(first_residual, gamma1, beta1)

    feed_forward_output = feed_forward(first_layer_norm, W1, b1, W2, b2)

    second_residual = first_layer_norm + feed_forward_output
    second_layer_norm = layer_norm(second_residual, gamma2, beta2)

    return second_layer_norm

In [ ]:
d_model = 4
d_ff = d_model * 4
num_tokens = 3

np.random.seed(42)
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model)
W1 = np.random.randn(d_model, d_ff)
b1 = np.random.randn(d_ff)
W2 = np.random.randn(d_ff, d_model)
b2 = np.random.randn(d_model)
gamma1 = np.ones(d_model)
beta1 = np.zeros(d_model)
gamma2 = np.ones(d_model)
beta2 = np.zeros(d_model)

X = np.array([[1,0,1,0],[0,1,0,1],[1,1,0,0]], dtype=float)

output = transformer_block(X, W_Q, W_K, W_V, W_O, W1, b1, W2, b2, gamma1, beta1, gamma2, beta2)
print(output)
print(output.shape)  # should be (3, 4)

[[ 0.62823365 -0.10740076 -1.78765202  0.86530829]
 [ 0.07980108  0.42927436 -1.46833184  0.24695146]
 [ 0.52508593 -0.07180934 -1.20964397  1.87018316]]
(3, 4)
